# 02. Baseline ASR (Speech-to-Text)

## Goal
Establish a baseline transcription workflow using an open-weight Thai-supported ASR model (OpenAI Whisper) on a small set of samples from the LOTUSDIS dataset.

## Why This Step Matters
Evaluating a pre-trained model on our dataset before any fine-tuning gives us a benchmark performance (Word Error Rate / Character Error Rate). This comparison tells us exactly how much value fine-tuning adds later.

## Inputs
- Streamed `nectec/LOTUSDIS` dataset from Hugging Face.
- Pre-trained `openai/whisper-tiny` model weights.

## Outputs
- Cleaned ASR transcriptions for a small test subset.
- Baseline error/performance logs for comparison against reference text.

## Setup

Let's import the necessary libraries. We will also import our custom `normalize_text` function to clean our text.

In [7]:
import torch
from transformers import pipeline
from datasets import load_dataset
import sys
from pathlib import Path

# Add src/ to the path so we can import our asr module
sys.path.append(str(Path("..").resolve()))
from src.asr.normalize_text import normalize_text

print("Setup completed successfully.")
print("CUDA Available:", torch.cuda.is_available())

Setup completed successfully.
CUDA Available: True


In [8]:
import sys
print(sys.executable)

/home/lataeq/Code/Saranote/.venv/bin/python


## Load Model

We will use the Hugging Face `pipeline` API to load `openai/whisper-tiny`. Whisper is a state-of-the-art multilingual model that includes support for Thai.

In [9]:
# Select device (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the ASR pipeline
# We set chunk_length_s to handle potentially long segments and specify language="thai"
print(f"Loading openai/whisper-tiny model on {device}...")
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=device,
    chunk_length_s=30,
)
print("ASR model loaded successfully.")

Loading openai/whisper-tiny model on cuda...


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 8840.14it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


ASR model loaded successfully.


## Load Data

We stream the `nectec/LOTUSDIS` dataset from Hugging Face and extract a few samples from the training split.

In [10]:
# Load dataset in streaming mode
dataset = load_dataset("nectec/LOTUSDIS", streaming=True)
train_split = dataset["train"]

# Fetch a few samples (let's do 5 samples)
num_samples = 5
samples = []
iterator = iter(train_split)

for _ in range(num_samples):
    samples.append(next(iterator))

print(f"Successfully loaded {len(samples)} samples.")

Successfully loaded 5 samples.


## Run Experiment (Transcription)

Now we run the Whisper model on each sample's audio array.
Note: Whisper expects the input audio dict to have `array` and `sampling_rate`.

In [11]:
results = []

for i, sample in enumerate(samples):
    print(f"\nTranscribing sample {i+1}/{num_samples}...")
    audio_data = sample["audio"]
    reference = sample["sentence"]  # The ground truth text
    
    # Run transcription
    # We specify generate_kwargs to force Whisper to decode into Thai
    prediction_output = asr_pipe(
        audio_data,
        generate_kwargs={"language": "thai", "task": "transcribe"}
    )
    prediction = prediction_output["text"]
    
    # Normalize texts
    normalized_ref = normalize_text(reference)
    normalized_pred = normalize_text(prediction)
    
    results.append({
        "raw_reference": reference,
        "raw_prediction": prediction,
        "norm_reference": normalized_ref,
        "norm_prediction": normalized_pred
    })

print("All samples transcribed successfully.")


Transcribing sample 1/5...

Transcribing sample 2/5...

Transcribing sample 3/5...

Transcribing sample 4/5...

Transcribing sample 5/5...
All samples transcribed successfully.


## Inspect Results

Let's compare the raw and normalized predictions against the reference texts.

In [12]:
for idx, res in enumerate(results):
    print(f"--- Sample {idx + 1} ---")
    print(f"Raw Reference:   {res['raw_reference']}")
    print(f"Raw Prediction:  {res['raw_prediction']}")
    print(f"Norm Reference:  {res['norm_reference']}")
    print(f"Norm Prediction: {res['norm_prediction']}")
    print()

--- Sample 1 ---
Raw Reference:   ครับสวัสดีครับ M ศูนย์หนึ่งครับผม
Raw Prediction:  ครับ ส่วนรึงครับผม
Norm Reference:  ครับสวัสดีครับ m ศูนย์หนึ่งครับผม
Norm Prediction: ครับ ส่วนรึงครับผม

--- Sample 2 ---
Raw Reference:   สวัสดีค่ะ F ศูนย์หนึ่งค่ะ
Raw Prediction:  สวัสดีค่ะ แฟฟสุดลึงค่ะ
Norm Reference:  สวัสดีค่ะ f ศูนย์หนึ่งค่ะ
Norm Prediction: สวัสดีค่ะ แฟฟสุดลึงค่ะ

--- Sample 3 ---
Raw Reference:   สวัสดีค่ะ F ศูนย์สองค่ะ
Raw Prediction:  สวัสดีค่ะ ฝฟุ้นสองค่ะ
Norm Reference:  สวัสดีค่ะ f ศูนย์สองค่ะ
Norm Prediction: สวัสดีค่ะ ฝฟุ้นสองค่ะ

--- Sample 4 ---
Raw Reference:   หัวข้อ Topic ที่ห้าสิบเจ็ดชอบกินข้าวนอกบ้านหรือทำกินเองที่บ้าน
Raw Prediction:  หวักค่ะ ทอบปิกที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ชอบกิน ข้าว นอกบ้าน เรื่องทำกินเองที่บ้าน
Norm Reference:  หัวข้อ topic ที่ห้าสิบเจ็ดชอบกินข้าวนอกบ้านหรือทำกินเองที่บ้าน
Norm Prediction: หวักค่ะ ทอบปิกที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ที่ ชอบกิน ข้าว นอกบ้าน เรื่องทำกินเองที่บ้าน

--- Sample 5 ---
Raw Reference:   ครับก็
Raw Predi